# Cluster wallets based on how they interact with contracts

In [ ]:
# 📊 Data Handling
import pandas as pd
import numpy as np

# 📈 Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

# ⚙️ Preprocessing & Scaling
from datetime import timedelta
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler

# 📈 Analysis
import scipy.stats as stats
from sklearn.feature_selection import VarianceThreshold
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
  
# 🧠 Modeling
from sklearn.cluster import KMeans, DBSCAN
import hdbscan
from sklearn.metrics import silhouette_score

In [ ]:
data = pd.read_csv('Transactions.csv')
data.head()

# Feature Engineering

In [ ]:
df = data.copy()
df['BLOCK_TIMESTAMP'] = pd.to_datetime(df['BLOCK_TIMESTAMP'])
df = df.sort_values(by=['FROM_ADDRESS', 'BLOCK_TIMESTAMP'])
df.head()

In [ ]:
df['BLOCK_TIMESTAMP'].min()

In [ ]:
df['BLOCK_TIMESTAMP'].max()

In [ ]:
df.shape

In [ ]:

df[df['FUNCTION_NAME'] == 'setApprovalForAll']

In [ ]:
latest_time = df['BLOCK_TIMESTAMP'].max()
last_30 = latest_time - timedelta(days=30)
last_7 = latest_time - timedelta(days=7)

df_30 = df[df['BLOCK_TIMESTAMP'] >= last_30]
df_7 = df[df['BLOCK_TIMESTAMP'] >= last_7]

def aggregate_features(df_subset, suffix):
    return df_subset.groupby('FROM_ADDRESS').agg(
        {
            'TX_HASH': 'nunique',
            'FUNCTION_NAME': pd.Series.nunique,
            'VALUE': 'mean',
            'TO_ADDRESS': pd.Series.nunique,
            'GAS_USED' : 'mean'
        }
    ).rename(columns={
        'TX_HASH': f'total_tx_count_{suffix}',
        'FUNCTION_NAME': f'unique_function_count_{suffix}',
        'VALUE': f'avg_value_sent_{suffix}',
        'TO_ADDRESS': f'contract_diversity_{suffix}',
        'GAS_USED': f'GAS_USED_{suffix}'
    })

agg_30 = aggregate_features(df_30, '30d')
agg_7 = aggregate_features(df_7, '7d')

df_features1 = pd.concat([agg_30, agg_7], axis=1).fillna(0).reset_index()


# avg_tx_per_day: total_tx_count / active_days
avg_tx_per_day = ((df.groupby(['FROM_ADDRESS'])['TX_HASH'].nunique()) / (df.groupby(['FROM_ADDRESS'])['BLOCK_TIMESTAMP'].nunique())).rename('avg_tx_per_day')


# total_value_sent: Sum of VALUE
total_value_sent = df.groupby(['FROM_ADDRESS'])['VALUE'].sum().rename('total_value_sent')


# active_days: Number of days they were active 
active_days = df.groupby(['FROM_ADDRESS'])['BLOCK_TIMESTAMP'].nunique().rename('active_days')


# first_last_tx_diff: Activity duration in days
first_last_tx_diff = ((df.groupby(['FROM_ADDRESS'])['BLOCK_TIMESTAMP'].max()) - (df.groupby(['FROM_ADDRESS'])['BLOCK_TIMESTAMP'].min())).rename('first_last_tx_diff')


# final data frame
df_features2 = pd.concat([
    avg_tx_per_day,
    total_value_sent,
    active_days,
    first_last_tx_diff
], axis=1).reset_index()


df_features = pd.merge(df_features1,df_features2, on='FROM_ADDRESS', how='inner')

In [ ]:
df_features['first_last_tx_diff'] = df_features['first_last_tx_diff'].dt.days
df_features

In [ ]:
fig = px.box(df_features, x='total_tx_count_30d', title='total_tx_count')
fig.update_layout(bargap=0.1)
fig.show()

# Feature Selecting

In [ ]:
#Pairplot (Scatter Matrix) 
sns.pairplot(df_features[[
    'unique_function_count', 'total_tx_count', 'avg_value_sent',
    'GAS_USED', 'contract_diversity'
]])

In [ ]:
# Correlation Matrix 
plt.figure(figsize=(8, 8))
sns.heatmap(df_features.corr(numeric_only=True), annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Correlation Matrix of Features")
plt.show()

In [ ]:
df_features_final = df_features.drop(['total_tx_count_7d','contract_diversity_30d','GAS_USED_30d',
                                     'active_days','unique_function_count_7d','avg_value_sent_7d'],axis=1)

df_features_final

# Feature Binning

In [ ]:
# check features one by one through hist & quartiles
fig = px.histogram(df_features, x=np.log10(df_features['total_value_sent'].replace(0, np.nan)).dropna(), nbins=50)
fig.update_layout(bargap=0.1)
fig.show()

#np.log10(df_features_final['avg_value_sent_30d'].replace(0, np.nan)).dropna()

In [ ]:
bin_series, bin_edges = pd.qcut(
    df_features['avg_value_sent_30d'],
    q=4,
    retbins=True,
    duplicates='drop'
)

print(f"Number of bins created: {len(bin_edges)-1}")

In [ ]:
# final bins
# 1
df_features_final['tx_count_30d_bins'] = pd.qcut(df_features_final['total_tx_count_30d'],q=4,duplicates='drop')
# 2
df_features_final['function_count_30d_bin'] = (df_features_final['unique_function_count_30d'] > 1).astype(int)
# 4
def bin_avg_value_sent(row):
    if row <= 0.1:
        return 'low'
    elif row <= 1:
        return 'medium'
    elif row <= 10:
        return 'high'
    else:
        return 'very_high'

df_features_final['value_sent_bin'] = df_features_final['avg_value_sent_30d'].apply(bin_avg_value_sent)


# 5
def bin_contract_diversity(val):
    if val == 1:
        return 'very_low'
    elif val <= 4:
        return 'low'
    elif val <= 10:
        return 'medium'
    elif val <= 30:
        return 'high'
    else:
        return 'very_high'

df_features_final['contract_diversity_7d_bin'] = df_features_final['contract_diversity_7d'].apply(bin_contract_diversity)


# 6
def bin_gas_used(val):
    if val <= 3000:
        return 'very_low'
    elif val <= 20000:
        return 'low'
    elif val <= 100000:
        return 'medium'
    elif val <= 1000000:
        return 'high'
    else:
        return 'very_high'

df_features_final['gas_used_7d_bin'] = df_features_final['GAS_USED_7d'].apply(bin_gas_used)



# 7
def bin_tx_per_day(val):
    if val == 0:
        return 'inactive'
    elif val <= 0.05:
        return 'very_low'
    elif val <= 0.2:
        return 'low'
    elif val <= 0.6:
        return 'medium'
    elif val <= 1.0:
        return 'high'
    else:
        return 'very_high'

df_features_final['tx_per_day_bin'] = df_features_final['avg_tx_per_day'].apply(bin_tx_per_day)


# 8
def bin_total_value(val):
    if val == 0:
        return 'none_or_zero'
    elif val <= 0.01:
        return 'tiny'
    elif val <= 0.1:
        return 'low'
    elif val <= 1:
        return 'moderate'
    elif val <= 10:
        return 'high'
    else:
        return 'very_high'

df_features_final['value_sent_bin'] = df_features_final['total_value_sent'].apply(bin_total_value)


# 9
bins = [0, 1, 3, 7, 14]
labels = ['very_short', 'short', 'medium', 'long']
df_features_final['first_last_bin'] = pd.cut(df_features_final['first_last_tx_diff'], bins=bins, labels=labels, include_lowest=True)

In [ ]:
df_features_final['tx_count_30d_bin_binary'] = df_features_final['tx_count_30d_bins'].apply(lambda x: 1 if x.right > 2 else 0)

df_features_final_bin = df_features_final.drop(['total_tx_count_30d','unique_function_count_30d',
                                               'avg_value_sent_30d','contract_diversity_7d','GAS_USED_7d','avg_tx_per_day',
                                               'total_value_sent','first_last_tx_diff','tx_count_30d_bins'],axis=1)
df_features_final_bin.head()

# Encoding + Scaling

In [ ]:
df_features_final_bin.info()

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from scipy.spatial.distance import pdist, squareform
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import silhouette_score

df_encoded = df_features_final_bin.copy()
for col in df_encoded.columns:
    if df_encoded[col].dtype == 'object' or isinstance(df_encoded[col][0], str):
        df_encoded[col] = LabelEncoder().fit_transform(df_encoded[col])


In [ ]:
df_encoded = df_encoded.drop(columns=["FROM_ADDRESS"])
df_encoded

# Clustering by Hierarchical

In [ ]:
# Calculate pairwise distances (Jaccard and Hamming)
dist_jaccard = pdist(df_encoded, metric='jaccard')
dist_hamming = pdist(df_encoded, metric='hamming')

#Hierarchical Clustering using linkage
linkage_jaccard = linkage(dist_jaccard, method='ward')  
linkage_hamming = linkage(dist_hamming, method='ward')

In [ ]:
# Plot Dendrograms
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
dendrogram(linkage_jaccard)
plt.title("Jaccard - Dendrogram")

plt.subplot(1, 2, 2)
dendrogram(linkage_hamming)
plt.title("Hamming - Dendrogram")
plt.tight_layout()
plt.show()

In [ ]:
from scipy.cluster.hierarchy import fcluster
labels_jaccard = fcluster(linkage_jaccard, t=6, criterion='maxclust')
df_features_final_bin['cluster_jaccard'] = labels_jaccard
df_features_final_bin

In [ ]:
for cluster_id, group in df_features_final_bin.groupby('cluster_jaccard'):
    print(f"Cluster {cluster_id}")
    print(f"Count: {len(group)}")
    print(group.drop(columns=['FROM_ADDRESS', 'cluster_jaccard']).mode().T)
    print("-" * 50)

In [ ]:
final_clustered_wallets = df_features_final_bin.copy()
final_clustered_wallets

In [ ]:
final_clustered_wallets.to_csv('data_all.csv')

# Clustering

In [ ]:
# Log Transformation for Skewed Features
df_transformed = df_features.copy()

cols_to_log = ['total_tx_count_30d', 'unique_function_count_30d',
       'avg_value_sent_30d', 'contract_diversity_30d', 'GAS_USED_30d',
       'total_tx_count_7d', 'unique_function_count_7d', 'avg_value_sent_7d',
       'contract_diversity_7d', 'GAS_USED_7d', 'avg_tx_per_day',
       'total_value_sent', 'active_days', 'first_last_tx_diff',
       'total_tx_count_30d_log']

for col in cols_to_log:
    df_transformed[col] = np.log1p(df_transformed[col]) 

In [ ]:
# Feature Scaling
X = df_transformed.drop(columns=['FROM_ADDRESS'])
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# DBSCAN Clustering

In [ ]:
from sklearn.cluster import DBSCAN

# Use PCA output for clustering (optional: use X_scaled directly for better accuracy)
dbscan = DBSCAN(eps=0.5, min_samples=50)
cluster_labels = dbscan.fit_predict(X_scaled)

# Add cluster labels to your dataframe
df_transformed['cluster_DBSCAN'] = cluster_labels

In [ ]:
df_transformed

In [ ]:
df_transformed['cluster_DBSCAN'].value_counts()

In [ ]:
cluster_summary = df_transformed.groupby('cluster_DBSCAN').mean()
cluster_summary

In [ ]:
cluster_mapping_DBSCAN = {
    3: "whale_wallets",
    5: "defi_farmers",
    11: "defi_farmers",
    9: "long_term_passive",
    10: "long_term_passive",
    0: "inactive_wallet",
    1: "inactive_wallet",
    2: "nft_or_airdrop_only",
    4: "nft_or_airdrop_only",
    6: "nft_or_airdrop_only",
    8: "nft_or_airdrop_only",
    14: "nft_or_airdrop_only",
    7: "bot_or_bet_like",
    13: "bot_or_bet_like",
    -1: "noise_or_misc"
}


df_transformed['DBSCAN_cluster'] = df_transformed['cluster_DBSCAN'].map(cluster_mapping_DBSCAN)

df_transformed

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
sns.scatterplot(
    x=X_scaled[:, 0], y=X_scaled[:, 1],
    hue=cluster_labels,
    palette="tab10",
    s=60,
    alpha=0.8,
    legend='full'
)
plt.title("DBSCAN Clustering Results (PCA Projection)")
plt.xlabel("PCA Component 1")
plt.ylabel("PCA Component 2")
plt.legend(title='Cluster')
plt.grid(True)
plt.show()

# HDBSCAN Clustering

In [ ]:
import hdbscan

In [ ]:
# Run HDBSCAN on scaled features (or PCA-transformed)
clusterer_HDBSCAN = hdbscan.HDBSCAN(
    min_cluster_size=10,        # Minimum cluster size
    min_samples=10,             # Core point threshold (optional)
    prediction_data=True       # Enables soft clustering
)

clusterer_HDBSCAN_labels = clusterer_HDBSCAN.fit_predict(X_scaled)

# Add to dataframe
df_transformed['cluster_HDBSCAN'] = cluster_labels

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
sns.scatterplot(
    x=X_scaled[:, 0], y=X_scaled[:, 1],
    hue=cluster_labels,
    palette='tab10',
    alpha=0.8,
    s=60,
    legend='full'
)
plt.title("HDBSCAN Clustering (PCA Projection)")
plt.xlabel("PCA Component 1")
plt.ylabel("PCA Component 2")
plt.legend(title="Cluster")
plt.grid(True)
plt.show()

In [ ]:
 df_features.hist() 

In [ ]:
df_transformed.groupby('cluster_HDBSCAN').mean(numeric_only=True)

In [ ]:
cluster_label_map = {
    3: 'whale_wallet',
    5: 'defi_farmer',
    11: 'defi_farmer',
    0: 'inactive_wallet',
    2: 'inactive_wallet',
    4: 'inactive_wallet',
    6: 'inactive_wallet',
    14: 'inactive_wallet',
    7: 'bot_like_user',
    12: 'airdrop_hunter',
    13: 'airdrop_hunter',
    1: 'long_term_passive',
    9: 'long_term_passive',
    10: 'long_term_passive',
    -1: 'noise'
}


df_transformed['HDBSCAN_cluster'] = df_transformed['cluster_HDBSCAN'].map(cluster_label_map)

In [ ]:
df_transformed

# K-means Clustering

In [ ]:
#  Elbow & Silhouette
inertia = []
silhouette_scores = []
k_range = range(2, 11)

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10) 
    labels = kmeans.fit_predict(X_pca)
    inertia.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(X_pca, labels))

fig, ax = plt.subplots(1, 2, figsize=(14, 5))

ax[0].plot(k_range, inertia, marker='o')
ax[0].set_title('Elbow Method - Inertia vs. K')
ax[0].set_xlabel('Number of Clusters (k)')
ax[0].set_ylabel('Inertia')

ax[1].plot(k_range, silhouette_scores, marker='o', color='green')
ax[1].set_title('Silhouette Score vs. K')
ax[1].set_xlabel('Number of Clusters (k)')
ax[1].set_ylabel('Silhouette Score')

plt.tight_layout()
plt.show()

In [ ]:
X = df_transformed.drop(columns=['FROM_ADDRESS'], axis=1) 
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# K=7
kmeans = KMeans(n_clusters=7, random_state=42, n_init='auto')
df_transformed['cluster_kmeans'] = kmeans.fit_predict(X_scaled)

In [ ]:
df_transformed['cluster_kmeans'].value_counts()

In [ ]:
df_transformed.groupby('cluster_kmeans').mean(numeric_only=True)

In [ ]:
df_transformed

In [ ]:
kmeans_cluster_map = {
    0: 'inactive_wallet',
    1: 'airdrop_hunter',
    2: 'defi_farmer',
    3: 'long_term_passive',
    4: 'bot_like_user',
    5: 'mini_whale',
    6: 'whale_wallet'
}

df_transformed['Kmeans_cluster'] = df_transformed['cluster_kmeans'].map(kmeans_cluster_map)

# Digram 

In [ ]:
median_days = df_features['active_days'].median()
df_features['active_days_high'] = (df_features['active_days'] > median_days).astype(int)

median_function = df_features['unique_function_count'].median()
df_features['function_high'] = (df_features['unique_function_count'] > median_function).astype(int)

median_tx = df_features['total_tx_count'].median()
df_features['tx_high'] = (df_features['total_tx_count'] > median_tx).astype(int)

In [ ]:
df_features

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram
from scipy.spatial.distance import pdist, squareform
import matplotlib.pyplot as plt


binary_features = ['active_days_high', 'function_high', 'tx_high']  
X_bin = df_features[binary_features]
X_bin = X_bin.fillna(0).astype(int)
from scipy.spatial.distance import pdist
from scipy.cluster.hierarchy import linkage, dendrogram
import matplotlib.pyplot as plt

jaccard_dist = pdist(X_bin, metric='jaccard')

linkage_matrix = linkage(jaccard_dist, method='average')  # average یا complete بهتره برای داده باینری

plt.figure(figsize=(16, 6))
dendrogram(
    linkage_matrix,
    truncate_mode='level',  # فقط n سطح اول
    p=12,                     # تعداد سطوح (مثلاً 5)
    leaf_rotation=90,
    leaf_font_size=6
)
plt.title("Truncated Dendrogram")
plt.show()

In [ ]:
from scipy.cluster.hierarchy import fcluster
clusters = fcluster(linkage_matrix, t=5, criterion='maxclust')  # مثلاً ۵ خوشه
df_features['cluster_hierarchical'] = clusters

In [ ]:
df_features

# Final Segmentation

In [ ]:
# Create a new column showing number of methods that agree
df_transformed['segment_agreement_count'] = (
    (df_transformed['DBSCAN_cluster'] == df_transformed['HDBSCAN_cluster']).astype(int) +
    (df_transformed['HDBSCAN_cluster'] == df_transformed['Kmeans_cluster']).astype(int) +
    (df_transformed['Kmeans_cluster'] == df_transformed['DBSCAN_cluster']).astype(int)
)

In [ ]:
df_transformed['segment_agreement_count'].value_counts()

In [ ]:
def choose_segment(row):
    if row['segment_agreement_count'] == 3:
        return row['DBSCAN_cluster']  # or any, since all agree
    elif row['HDBSCAN_cluster'] != 'noise':
        return row['HDBSCAN_cluster']
    elif row['Kmeans_cluster'] != 'inactive_wallet':
        return row['Kmeans_cluster']
    else:
        return row['DBSCAN_cluster']

df_transformed['final_user_segment'] = df_transformed.apply(choose_segment, axis=1)

In [ ]:
import seaborn as sns
import pandas as pd

confusion_matrix = pd.crosstab(df_transformed['DBSCAN_cluster'], df_transformed['Kmeans_cluster'])
sns.heatmap(confusion_matrix, annot=True, fmt="d", cmap="YlGnBu")
plt.title("DBSCAN vs KMeans Segment Agreement")
plt.show()

In [ ]:
df_transformed.head(1)

In [ ]:
confusion_matrix = pd.crosstab(df_transformed['HDBSCAN_cluster'], df_transformed['DBSCAN_cluster'])
sns.heatmap(confusion_matrix, annot=True, fmt="d", cmap="YlGnBu")
plt.title("HDBSCAN vs DBSCAN Segment Agreement")
plt.show()

In [ ]:
confusion_matrix = pd.crosstab(df_transformed['HDBSCAN_cluster'], df_transformed['Kmeans_cluster'])
sns.heatmap(confusion_matrix, annot=True, fmt="d", cmap="YlGnBu")
plt.title("HDBSCAN vs KMeans Segment Agreement")
plt.show()

In [ ]:
df_transformed.groupby('final_user_segment').mean(numeric_only=True)

In [ ]:
def decide_final_segment(row):
    dbscan = row['DBSCAN_cluster']
    hdbscan = row['HDBSCAN_cluster']
    kmeans = row['Kmeans_cluster']

    if dbscan == hdbscan == kmeans:
        return dbscan

    if hdbscan != 'noise' and hdbscan != 'noise_or_misc':
        return hdbscan

    if hdbscan in ['noise', 'noise_or_misc'] and kmeans != 'inactive_wallet':
        return kmeans

    return dbscan

df_transformed['final_user_segment'] = df_transformed.apply(decide_final_segment, axis=1)

In [ ]:
df_transformed['final_user_segment'].value_counts()

In [ ]:
round((13217/15514)*100)

In [ ]:
df_transformed['final_user_segment'].count()

# Model 4: Clustering by functions

In [ ]:
# trace data 
test1 = pd.read_csv('data/01.csv',on_bad_lines='skip')
test2 = pd.read_csv('data/02.csv',on_bad_lines='skip')
test3 = pd.read_csv('data/03.csv',on_bad_lines='skip')
test4 = pd.read_csv('data/04.csv',on_bad_lines='skip')
test5 = pd.read_csv('data/05.csv',on_bad_lines='skip')
test6 = pd.read_csv('data/06.csv',on_bad_lines='skip')
test7 = pd.read_csv('data/07.csv',on_bad_lines='skip')
test8 = pd.read_csv('data/08.csv',on_bad_lines='skip')
test9 = pd.read_csv('data/09.csv',on_bad_lines='skip')
test10 = pd.read_csv('data/10.csv',on_bad_lines='skip')
test11 = pd.read_csv('data/11.csv',on_bad_lines='skip')
test12 = pd.read_csv('data/12.csv',on_bad_lines='skip')


df4 = pd.concat([test1,test2,test3,test4,test5,test6,test7,test8,test9,test10,
                test11,test12])
#df4['block_timestamp'] = pd.to_datetime(df4['block_timestamp'])
#df4 = pd.read_csv('04_df_row.csv',on_bad_lines='skip')

In [ ]:
# remove the wallets which limited tx

df4['BLOCK_TIMESTAMP'] = df4['BLOCK_TIMESTAMP'].astype(str).str.strip().str.replace(r'[^\x00-\x7F]+', '', regex=True)
df4 = df4[pd.to_datetime(df4['BLOCK_TIMESTAMP'], errors='coerce').notna()]
df4['BLOCK_TIMESTAMP'] = pd.to_datetime(df4['BLOCK_TIMESTAMP'])
df4['YEAR_MONTH'] = df4['BLOCK_TIMESTAMP'].dt.to_period('M')
tx_counts = (
    df4
    .groupby(['FROM_ADDRESS', 'YEAR_MONTH'])
    .size()
    .reset_index(name='TX_COUNT'))



# Final wallet list
wallets = tx_counts[tx_counts['TX_COUNT'] > 12]
valid_wallets = wallets['FROM_ADDRESS'].unique()
df4_filtered = df4[df4['FROM_ADDRESS'].isin(valid_wallets)]




In [ ]:
# behavior vector
function_vector = (
    df4_filtered
    .groupby(['FROM_ADDRESS', 'FUNCTION_NAME'])
    .size()
    .unstack(fill_value=0)
    .reset_index())

In [ ]:
user_profiles = {
    "DEX_TRADER": [
        'swap', 'swapExactTokensForETHSupportingFeeOnTransferTokens', 'swapExactETHForTokens',
        'swapExactTokensForTokens', 'unoswap', 'getAmountsOut', 'getReserves',
        'approve', 'transfer', 'balanceOf', 'allowance', 'WETH'
    ],
    "YIELD_FARMER": [
        'deposit', 'withdraw', 'collect', 'claimTokens', 'claimMintRewardAndShare',
        'callClaimMintReward', 'mint', 'burn', 'getTier', 'transformERC20'
    ],
    "STAKER_VALIDATOR": [
        'powerDown', 'completeQueuedWithdrawals', 'proveBlock', 'proposeBlock',
        'claimMintRewardAndShare', 'callClaimMintReward'
    ],
    "NFT_COLLECTOR": [
        'safeTransferFrom', 'ownerOf', 'mint', 'burn', 'claimRank'
    ],
    "AIRDROP_HUNTER": [
        'claimTokens', 'claimMintRewardAndShare', 'callClaimMintReward', 'claimRank', 'approve'
    ],
    "BRIDGE_USER": [
        'publishMessage', 'verifyProof', 'messageFee', 'resolve', 'getProvider'
    ],
    "PROTOCOL_DEV": [
        'execute', 'executeDelegateCall', 'getApp', 'implementation', 'router', 'transform',
        'token0', 'token1'
    ],
    "ORACLE_USER": [
        'latestRoundData', 'latestAnswer', 'slot0', 'getSinglePrice', 'status', 'getTier', 'decimals'
    ],
    "DEFI_FARMER": [
        'deposit', 'withdraw', 'collect', 'claimTokens', 'claimMintRewardAndShare',
        'callClaimMintReward', 'transformERC20', 'getReserves', 'swap', 'approve', 'mint', 'burn'
    ],
    "BOT": [
        'execute', 'executeDelegateCall', 'permit', 'transformERC20', 'resolve', 'router',
        'unoswap', 'getAmountsOut', 'latestAnswer', 'slot0', 'getSinglePrice', 'messageFee', 'publishMessage'
    ]
}



for func in {f for funcs in user_profiles.values() for f in funcs}:
    if func not in function_vector.columns:
        function_vector[func] = 0

for profile_name, function_list in user_profiles.items():
    function_vector[f"{profile_name}_SCORE"] = function_vector[function_list].sum(axis=1)

score_columns = ['FROM_ADDRESS'] + [f"{profile}_SCORE" for profile in user_profiles]
function_vector_profile_scores = function_vector[score_columns]

In [ ]:
profile_score_columns = [col for col in function_vector_profile_scores.columns if col.endswith("_SCORE") and col != "from_address"]

function_vector_profile_scores["TOP_PROFILE"] = function_vector_profile_scores[profile_score_columns].idxmax(axis=1)

function_vector_profile_scores["TOP_PROFILE_SCORE"] = function_vector_profile_scores[profile_score_columns].max(axis=1)

function_vector_profile_scores["TOP_PROFILE"] = (
    function_vector_profile_scores["TOP_PROFILE"]
    .str.replace("_SCORE", "", regex=False)
    .str.replace("_", " ")
    .str.title()
)

df_top_profiles = function_vector_profile_scores[["FROM_ADDRESS", "TOP_PROFILE", "TOP_PROFILE_SCORE"]]

#df_top_profiles.head()

In [ ]:
print("priamry wallets",df4['FROM_ADDRESS'].nunique())
print("after filtering",df4_filtered['FROM_ADDRESS'].nunique())

In [ ]:
df_top_profiles['TOP_PROFILE'].value_counts()
#df_top_profiles.to_csv('04_df_final.csv')

## Line Charts

In [ ]:
df5 = pd.merge(df4_filtered, df_top_profiles, on="FROM_ADDRESS", how="left")

In [ ]:
df5.head()

In [ ]:
# line chart - tx count 
df5['BLOCK_TIMESTAMP'] = df5['BLOCK_TIMESTAMP'].astype(str).str.strip()
df5['BLOCK_TIMESTAMP'] = df5['BLOCK_TIMESTAMP'].str.replace('\ufeff', '', regex=True)
df5['BLOCK_TIMESTAMP'] = pd.to_datetime(df5['BLOCK_TIMESTAMP'], errors='coerce')
df5 = df5.dropna(subset=['BLOCK_TIMESTAMP'])
df5['MONTH'] = df5['BLOCK_TIMESTAMP'].dt.to_period('M')

#df5['MONTH'] = pd.to_datetime(df5['BLOCK_TIMESTAMP']).dt.to_period('M')
category_total_tx = df5.groupby(['TOP_PROFILE', 'MONTH']) \
    .size().reset_index(name='TOTAL_TX')

category_wallet_counts = df5.groupby(['TOP_PROFILE', 'MONTH'])['FROM_ADDRESS'] \
    .nunique().reset_index(name='WALLET_COUNT')

category_avg_tx = pd.merge(category_total_tx, category_wallet_counts, on=['TOP_PROFILE', 'MONTH'])
category_avg_tx['CATEGORY_TX_MEAN'] = category_avg_tx['TOTAL_TX'] / category_avg_tx['WALLET_COUNT']

category_avg_tx = category_avg_tx[['TOP_PROFILE', 'MONTH', 'CATEGORY_TX_MEAN']]

wallet_monthly_tx = df5.groupby(['FROM_ADDRESS', 'TOP_PROFILE', 'MONTH']) \
    .size().reset_index(name='WALLET_TX_COUNT')

result = pd.merge(wallet_monthly_tx, category_avg_tx, on=['TOP_PROFILE', 'MONTH'], how='left')

result = result[['MONTH', 'FROM_ADDRESS', 'TOP_PROFILE', 'WALLET_TX_COUNT', 'CATEGORY_TX_MEAN']]

result['CATEGORY_TX_MEAN'] = round(result['CATEGORY_TX_MEAN'])
#result.to_csv('line_chart.csv')

In [ ]:
# line chart -  GAS_USED
category_total_gas = df5.groupby(['TOP_PROFILE', 'MONTH'])['GAS_USED'] \
    .sum().reset_index(name='TOTAL_GAS_USED')

category_wallets_gas = df5.groupby(['TOP_PROFILE', 'MONTH'])['FROM_ADDRESS'] \
    .nunique().reset_index(name='WALLET_COUNT')

category_avg_gas = pd.merge(category_total_gas, category_wallets_gas, on=['TOP_PROFILE', 'MONTH'])
category_avg_gas['CATEGORY_GAS_MEAN'] = category_avg_gas['TOTAL_GAS_USED'] / category_avg_gas['WALLET_COUNT']
category_avg_gas = category_avg_gas[['TOP_PROFILE', 'MONTH', 'CATEGORY_GAS_MEAN']]

wallet_monthly_gas = df5.groupby(['FROM_ADDRESS', 'TOP_PROFILE', 'MONTH'])['GAS_USED'] \
    .sum().reset_index(name='WALLET_GAS_USED')

result_gas = pd.merge(wallet_monthly_gas, category_avg_gas, on=['TOP_PROFILE', 'MONTH'], how='left')
result_gas = result_gas[['MONTH', 'FROM_ADDRESS', 'TOP_PROFILE', 'WALLET_GAS_USED', 'CATEGORY_GAS_MEAN']]

In [ ]:
# line chart - VALUE

category_total_value = df5.groupby(['TOP_PROFILE', 'MONTH'])['VALUE'] \
    .sum().reset_index(name='TOTAL_VALUE')

category_wallets_value = df5.groupby(['TOP_PROFILE', 'MONTH'])['FROM_ADDRESS'] \
    .nunique().reset_index(name='WALLET_COUNT')

category_avg_value = pd.merge(category_total_value, category_wallets_value, on=['TOP_PROFILE', 'MONTH'])
category_avg_value['CATEGORY_VALUE_MEAN'] = category_avg_value['TOTAL_VALUE'] / category_avg_value['WALLET_COUNT']
category_avg_value = category_avg_value[['TOP_PROFILE', 'MONTH', 'CATEGORY_VALUE_MEAN']]

wallet_monthly_value = df5.groupby(['FROM_ADDRESS', 'TOP_PROFILE', 'MONTH'])['VALUE'] \
    .sum().reset_index(name='WALLET_TOTAL_VALUE')

result_value = pd.merge(wallet_monthly_value, category_avg_value, on=['TOP_PROFILE', 'MONTH'], how='left')
result_value = result_value[['MONTH', 'FROM_ADDRESS', 'TOP_PROFILE', 'WALLET_TOTAL_VALUE', 'CATEGORY_VALUE_MEAN']]

## Category Card

In [ ]:
df5['BLOCK_TIMESTAMP'] = pd.to_datetime(df5['BLOCK_TIMESTAMP'], errors='coerce')

df5['YEAR_MONTH'] = df5['BLOCK_TIMESTAMP'].dt.to_period('M')

monthly_tx = (
    df5.groupby(['TOP_PROFILE', 'YEAR_MONTH'])['TX_HASH']
    .nunique()
    .reset_index(name='tx_count')
)

avg_tx_per_month = (
    monthly_tx.groupby('TOP_PROFILE')['tx_count']
    .mean()
    .reset_index(name='avg_tx_per_month')
)

unique_functions = (
    df5.groupby('TOP_PROFILE')['FUNCTION_NAME']
    .nunique()
    .reset_index(name='unique_function_count')
)

unique_contracts = (
    df5.groupby('TOP_PROFILE')['TO_ADDRESS']
    .nunique()
    .reset_index(name='unique_contract_count')
)

avg_gas_used = (
    df5.groupby('TOP_PROFILE')['GAS_USED']
    .mean()
    .reset_index(name='avg_gas_used')
)

metrics_df = avg_tx_per_month \
    .merge(unique_functions, on='TOP_PROFILE') \
    .merge(unique_contracts, on='TOP_PROFILE') \
    .merge(avg_gas_used, on='TOP_PROFILE')

metrics_df['avg_tx_per_month'] = round(metrics_df['avg_tx_per_month'])
metrics_df['avg_gas_used'] = round(metrics_df['avg_gas_used'])


## Heatmap

In [ ]:
df5['BLOCK_TIMESTAMP'] = pd.to_datetime(df5['BLOCK_TIMESTAMP'])

df5['WEEKDAY'] = df5['BLOCK_TIMESTAMP'].dt.day_name()
df5['HOUR'] = df5['BLOCK_TIMESTAMP'].dt.hour

weekday_activity = (
    df5.groupby(['TOP_PROFILE', 'WEEKDAY'])['FROM_ADDRESS']
    .nunique()
    .reset_index(name='UNIQUE_WALLETS')
)

weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
weekday_activity['WEEKDAY'] = pd.Categorical(weekday_activity['WEEKDAY'], categories=weekday_order, ordered=True)

hourly_activity = (
    df5.groupby(['TOP_PROFILE', 'HOUR'])['FROM_ADDRESS']
    .nunique()
    .reset_index(name='UNIQUE_WALLETS')
)

In [ ]:
weekday_activity.to_csv('weekday_activity.csv')
hourly_activity.to_csv('hourly_activity.csv')

# compare

In [ ]:
persona = pivoted_counts.drop(['allowance', 'approve', 'balanceOf', 'deposit',
       'execute', 'getReserves', 'latestAnswer', 'latestRoundData', 'swap',
       'token0', 'token1', 'transfer', 'transferFrom', 'uniswapV3SwapCallback',
       'withdraw', 'persona', 'DeFi_sign', 'defi_farmer_score', 'whale_score',
       'airdrop_hunter_score', 'bot_like_score', 'inactive_score',
       'long_term_passive_score'],axis=1)

persona

In [ ]:
feat = df_features.drop([ 'pca1', 'pca2', 'active_days_high',
       'function_high', 'tx_high', 'cluster_hierarchical'],axis=1)
feat

In [ ]:
df_co = pd.merge(feat, persona, on='FROM_ADDRESS', how='left')
df_co

In [ ]:
df_co.groupby('predicted_persona').mean(numeric_only=True)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 9 ویژگی مورد نظر برای قرار گرفتن در شبکه 3x3
features = [
    'unique_function_count', 'total_tx_count', 'avg_value_sent',
    'total_value_sent', 'GAS_USED', 'gas_efficiency',
    'active_days', 'contract_diversity', 'avg_tx_per_day'
]

# ساخت شکل و محورها
fig, axes = plt.subplots(3, 3, figsize=(18, 12))
axes = axes.flatten()

# رسم هر ویژگی در یک subplot
for i, feature in enumerate(features):
    sns.barplot(
        data=df_co,
        x='predicted_persona',
        y=feature,
        ci=None,
        estimator='mean',
        ax=axes[i],
        palette='viridis'
    )
    axes[i].set_title(f'{feature}', fontsize=12)
    axes[i].tick_params(axis='x', rotation=45)
    axes[i].set_xlabel('')
    axes[i].set_ylabel('Mean')

plt.tight_layout()
plt.show()

# Next part

In [ ]:
from scipy.stats import kruskal

features = [
    'unique_function_count', 'total_tx_count', 'avg_value_sent',
    'total_value_sent', 'GAS_USED', 'gas_efficiency',
    'active_days', 'contract_diversity', 'avg_tx_per_day'
]

print("Kruskal-Wallis test (non-parametric ANOVA):\n")
for feature in features:
    groups = [df_transformed[df_transformed['cluster_HDBSCAN'] == c][feature].dropna() for c in df_transformed['cluster_HDBSCAN'].unique()]
    stat, p = kruskal(*groups)
    print(f"{feature:25}: p-value = {p:.4f} → {'✔️ Significant' if p < 0.05 else '❌ Not Significant'}")

In [ ]:
from scipy.stats import kruskal

features = [
    'unique_function_count', 'total_tx_count', 'avg_value_sent',
    'total_value_sent', 'GAS_USED', 'gas_efficiency',
    'active_days', 'contract_diversity', 'avg_tx_per_day'
]

print("Kruskal-Wallis test (non-parametric ANOVA):\n")
for feature in features:
    groups = [df_transformed[df_transformed['cluster_DBSCAN'] == c][feature].dropna() for c in df_transformed['cluster_DBSCAN'].unique()]
    stat, p = kruskal(*groups)
    print(f"{feature:25}: p-value = {p:.4f} → {'✔️ Significant' if p < 0.05 else '❌ Not Significant'}")

In [ ]:
from scipy.stats import kruskal

features = [
    'unique_function_count', 'total_tx_count', 'avg_value_sent',
    'total_value_sent', 'GAS_USED', 'gas_efficiency',
    'active_days', 'contract_diversity', 'avg_tx_per_day'
]

print("Kruskal-Wallis test (non-parametric ANOVA):\n")
for feature in features:
    groups = [df_transformed[df_transformed['cluster_kmeans'] == c][feature].dropna() for c in df_transformed['cluster_kmeans'].unique()]
    stat, p = kruskal(*groups)
    print(f"{feature:25}: p-value = {p:.4f} → {'✔️ Significant' if p < 0.05 else '❌ Not Significant'}")

In [ ]:
from sklearn.metrics import silhouette_score

X = df_transformed[features]
labels = df_transformed['cluster_HDBSCAN']

score = silhouette_score(X, labels)
print(f'Silhouette Score: {score:.4f}')

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

for feature in features:
    plt.figure(figsize=(12, 4))
    sns.boxplot(data=df_transformed, x='cluster_HDBSCAN', y=feature)
    plt.title(f'{feature} per Cluster')
    plt.show()

# Part 2

In [ ]:
df2 = df.copy()

In [ ]:
persona_signatures = {
    "Airdrop Hunter": [
        "claimTokens", "mint", "permit", "claim", "claimRank"
    ],
    "DeFi Farmer": [
        "deposit", "withdraw", "stake", "harvest", "WETH", "allowance",
        "collect", "positions", "totalUnderlyingSupply", "getReserveNormalizedIncome", 
        "totalShares", "burn", "removeShares", "flushForwarderTokens", 
        "flushTokens", "getVaultWeight"
    ],
    "NFT User": [
        "mint", "transferFrom", "setApprovalForAll", "ownerOf", "burn", "isCursed"
    ],
    "Bot Trader": [
        "swap", "swapExactTokensForETHSupportingFeeOnTransferTokens", "getReserves",
        "uniswapV3SwapCallback", "router", "token0", "token1", "max", "min",
        "latestRoundData", "latestAnswer", "resolve", "transform", "fee",
        "getPair", "slot0", "getAmountsOut", "getSinglePrice", "fillOrderArgs", 
        "fulfillOrder", "tickBitmap", "observe", "multicall", "trade", 
        "quoteDeliveryPrice", "quoteEVMDeliveryPrice", "exactInputSingle", "exactInput", 
        "factory", "exchange", "getTakingAmount", "getAssetPrice", "getPrice", "getPriceInEth"
    ],
    "Protocol Power User": [
        "delegate", "vote", "propose", "executeProposal", "execute", "castVote",
        "getApp", "powerDown", "proveBlock", "implementation", "getProvider",
        "chainId", "execTransaction", "send", "sendMessage", "isSolver", "isValidator",
        "getInterfaceImplementer", "submitBatchSpendingReport", "enqueueSequencerMessage",
        "delayedMessageCount", "getContract", "postInteraction", "canCall", 
        "isBlacklisted", "isOperatorAllowed", "getUint", "status", "verifyProof",
        "publishMessage", "handleAction"
    ],
    "Retail Holder": [
        "transfer", "balanceOf", "approve", "allowance", "totalSupply", "decimals"
    ]
}

def detect_personas_multi(function_name):
    matched_personas = []
    for persona, signatures in persona_signatures.items():
        for sig in signatures:
            if sig.lower() in str(function_name).lower():
                matched_personas.append(persona)
    if matched_personas:
        return matched_personas
    else:
        return ["Other"]

In [ ]:
df2["persona_category"] = df2["FUNCTION_NAME"].apply(detect_personas_multi)


In [ ]:
df2

In [ ]:
df2['persona_category'].value_counts()

In [ ]:
df3 = df.copy()

persona_signatures = {
    "Airdrop Hunter": ["claimTokens", "mint", "permit", "claim", "claimRank"],
    "DeFi Farmer": ["deposit", "withdraw", "stake", "harvest", "WETH", "allowance",
                    "collect", "positions", "totalUnderlyingSupply", "getReserveNormalizedIncome",
                    "totalShares", "burn", "removeShares", "flushForwarderTokens",
                    "flushTokens", "getVaultWeight"],
    "NFT User": ["mint", "transferFrom", "setApprovalForAll", "ownerOf", "burn", "isCursed"],
    "Bot Trader": ["swap", "swapExactTokensForETHSupportingFeeOnTransferTokens", "getReserves",
                   "uniswapV3SwapCallback", "router", "token0", "token1", "max", "min",
                   "latestRoundData", "latestAnswer", "resolve", "transform", "fee",
                   "getPair", "slot0", "getAmountsOut", "getSinglePrice", "fillOrderArgs",
                   "fulfillOrder", "tickBitmap", "observe", "multicall", "trade",
                   "quoteDeliveryPrice", "quoteEVMDeliveryPrice", "exactInputSingle",
                   "exactInput", "factory", "exchange", "getTakingAmount", "getAssetPrice",
                   "getPrice", "getPriceInEth"],
    "Protocol Power User": ["delegate", "vote", "propose", "executeProposal", "execute", "castVote",
                             "getApp", "powerDown", "proveBlock", "implementation", "getProvider",
                             "chainId", "execTransaction", "send", "sendMessage", "isSolver",
                             "isValidator", "getInterfaceImplementer", "submitBatchSpendingReport",
                             "enqueueSequencerMessage", "delayedMessageCount", "getContract",
                             "postInteraction", "canCall", "isBlacklisted", "isOperatorAllowed",
                             "getUint", "status", "verifyProof", "publishMessage", "handleAction"],
    "Retail Holder": ["transfer", "balanceOf", "approve", "allowance", "totalSupply", "decimals"]
}

def calculate_persona_scores(function_list):
    scores = {persona: 0 for persona in persona_signatures.keys()}
    total_functions = len(function_list)

    if total_functions == 0:
        return scores       

    for func in function_list:
        for persona, signatures in persona_signatures.items():
            if any(sig.lower() in str(func).lower() for sig in signatures):
                scores[persona] += 1

    for persona in scores:
        scores[persona] = round(100 * scores[persona] / total_functions, 2)

    return scores


wallet_scores = []

for wallet, group in df3.groupby('FROM_ADDRESS'):
    functions = group['FUNCTION_NAME'].tolist()
    persona_score = calculate_persona_scores(functions)
    persona_score['FROM_ADDRESS'] = wallet
    wallet_scores.append(persona_score)

df_persona_scores = pd.DataFrame(wallet_scores)

In [ ]:
df_persona_scores

In [ ]:
persona_columns = ['Airdrop Hunter', 'DeFi Farmer', 'NFT User', 'Bot Trader', 'Protocol Power User', 'Retail Holder']

def get_dominant_persona(row):
    max_value = -1
    dominant = "Other"
    for persona in persona_columns:
        if row[persona] > max_value:
            max_value = row[persona]
            dominant = persona
    return dominant

df_persona_scores['dominant_persona'] = df_persona_scores.apply(get_dominant_persona, axis=1)


In [ ]:
df_persona_scores

# final persona

In [ ]:
df_merged = df2.merge(df_persona_scores[['FROM_ADDRESS', 'dominant_persona']], on='FROM_ADDRESS', how='left')

In [ ]:
df_merged

In [ ]:
df_merged['dominant_persona'].value_counts()

In [ ]:
df_merged.info()

In [ ]:
df_merged.head()